# Benchmarking to BIOCHLOR

This notebook provides a comparison of concentrations calculated with the *mibitrans* package and concentrations calculated with the *Excel*-based distribution of *BIOCHLOR* (Aziz et al., 2000). The notebook benchmarks the implementation of chain decay in *mibitrans*.

**Authors:** Jorrit Bakker

## Background

*BIOCHLOR* implements consecutive degradation of contaminants, specifically for chlorinated solvents. The Domenico (1987) solution is used, with the additional term (the same as the anatrans solution of this package). Decoupling of the consecutive degradation reactions is done by the methods described in Sun et al. (1999) (see documentation section for chain decay). In the *mibitrans* package, any transport model can be used, using the same transformation procedure as BIOCHLOR.

### BIOCHLOR example dataset and BIOCHLOR results

We use the set of transport and model parameters provided when initially opening BIOCHLOR for comparison. We ran BIOCHLOR in Excel to retrieve concentrations for comparison.

Sources:

Aziz, C. E., Newell, C. J., Gonzales, J. R., Haas, P., Clement, T. P., Sun, Y., & Jewett, D. G. (2000). BIOCHLOR natural attenuation decision support system. User’s Manual Version, 1.

Domenico, P. A., 1987, An analytical model for multidimensional transport of a decaying contaminant
species, Journal of Hydrology, 91 (1), 49–58.](https://doi.org/10.1016/0022-1694(87)90127-2)

Sun, Y., Petersen, J. N., & Clement, T. P. (1999). Analytical solutions for multiple species reactive transport in multiple dimensions. Journal of Contaminant Hydrology, 35(4), 429–440. https://doi.org/10.1016/S0169-7722(98)00105-3


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import mibitrans as mbt

In [ ]:
ft = 3.281

### Parameters and model

Use example parameters for perchloroethene (PCE) to trichloroethene (TCE), dichloroethene (DCE), vinyl chloride (VC) and lastly ethene (ETH). Provided in the same order for each parameter requiring entry for each compound separately.

In [ ]:
hydro = mbt.HydrologicalParameters(
    velocity = 111.741732283465 /365 / ft, # [m]
    porosity=0.2,  # [-]
    alpha_x=40 / ft,  # [m]
    alpha_y=4 / ft,  # [m]
    alpha_z=0,  # [m]
)

att = mbt.AttenuationParameters(
    retardation=2.872, # [-]
    decay_rate=np.array([2 / 365, 1 / 365, 0.7 / 365, 0.4 / 365, 0.000000001 / 365]) # [1/d]
)

source = mbt.SourceParameters(
    source_zone_boundary = 105/2/ft, # [m]
    source_zone_concentration = [0.056, 15.8, 98.5, 3.08, 0.03], #[g/m3]
    depth = 56 / ft, # [m]
    total_mass = np.inf # [g]
)

model = mbt.ModelParameters(
    model_length=1085 / ft,  # [m]
    model_width=280*2 / ft,  # [m]
    model_time=33 * 365,  # [days]
    dx=5,  # [m]
    dy=2,  # [m]
    dt=365,  # [days]
)

In [ ]:
biochlor_model = mbt.Anatrans(hydro, att, source, model)
biochlor_model.chain_decay(mass_ratios=np.array([0.79, 0.74, 0.64, 0.45]))
biochlor_results = biochlor_model.run()

### Load BIOCHLOR data

We ran BIOCHLOR with the specified parameters and stored calculated values in a `json`-file (`mibitrans.data.example_data.json`). It is accessed through the `BiochlorData` class.
We define the observation points in $x$, $y$ and $t$ as separate arrays.

In [ ]:
biochlor_data = mbt.BiochlorData()
x = biochlor_data.x / ft
y = biochlor_data.y / ft
t = biochlor_data.t * 365

### Visual comparison

In [ ]:
compounds=["pce", "tce", "dce", "vc", "eth"]
names=["PCE", "TCE", "DCE", "VC", "ETH"]
colors=["blue", "orange", "green", "red", "purple"]
kwargs_line = dict(lw=4, alpha=0.6)
kwargs_scatter = dict(marker='+', s=80, zorder=3)
biochlor_results.centerline(time=t[0], legend_names=names, **kwargs_line)
for i, comp in enumerate(compounds):
    plt.scatter(x, getattr(biochlor_data, comp)[0, 2, :], color=colors[i], **kwargs_scatter)
plt.show()

biochlor_results.centerline(time=t[1], legend_names=names, **kwargs_line)
for i, comp in enumerate(compounds):
    plt.scatter(x, getattr(biochlor_data, comp)[1, 2, :], color=colors[i], **kwargs_scatter)
plt.show()

biochlor_results.centerline(time=t[2], legend_names=names, **kwargs_line)
for i, comp in enumerate(compounds):
    plt.scatter(x, getattr(biochlor_data, comp)[-1, 2, :], color=colors[i], **kwargs_scatter)
plt.show()


### Numerical comparison

In [ ]:
model_coarse = mbt.ModelParameters(
    model_length=1085 / ft,  # [m]
    model_width=224*2 / ft,  # [m]
    model_time=33 * 365,  # [days]
    dx=108.5/ft,  # [m]
    dy=112/ft,  # [m]
    dt=365,  # [days]
)

In [ ]:
biochlor_model_coarse = mbt.Anatrans(hydro, att, source, model_coarse)
biochlor_model_coarse.chain_decay(mass_ratios=np.array([0.794918330308530,
                                                        0.737442922374429,
                                                        0.644994840041280,
                                                        0.449600000000000]))
biochlor_results_coarse = biochlor_model_coarse.run()

In [ ]:
a = getattr(biochlor_data, "tce")[1,:,:]
b = biochlor_results_coarse.cxyt[1][14,:,:]
print(a)
print(b)
print(abs(a-np.round(b, decimals=3)))


In [ ]:
from mibitrans.analysis import differences as diff

In [ ]:
time_index = [4, 14, -1]
compounds=["pce", "tce", "dce", "vc", "eth"]
for i, comp in enumerate(compounds):
    dff = diff.mean_absolute_difference(
        getattr(biochlor_data, comp), np.round(biochlor_results_coarse.cxyt[i][time_index,:,:], decimals=3)
    )
    print("Mean absolute difference in relative concentration between BIOCHLOR "
      f"and Anatrans model class for {comp}:", dff/np.max(biochlor_results_coarse.cxyt[i]))

In conclusion, the implementation of chain decay into mibitrans is the same as in BIOCHLOR.